In [1]:
import torch
import qewton
import plotly.io as pio

from load_data import (
    domain_var,
    temperature_var,
    x_var,
    load_data,
)

pio.renderers.default = "vscode"

In [2]:
batch_size = qewton.optim.DiscreteHyperparameter((5, 50))

dataset, data_loader = load_data(batch_size)


--- Data quality report ---
Total samples:     581
Valid samples:     581
Discarded samples: 0


In [3]:
model = qewton.algorithms.CNNDeepONet(
    branch_input=domain_var,
    trunk_input=x_var,
    output=temperature_var,

    # CNN branch
    channels=(16, 32),
    conv_kernel_size=(3, 3, 3),
    pooling_kernel_size=2,
    pooling_type="average",

    # Trunk network
    trunk_hidden_neurons=64,
    trunk_hidden_layers=3,

    # Shared representation dimension
    intermediate_neurons=64,

    # FCN after convolutional encoder
    branch_fcn_hidden_neurons=64,
    branch_fcn_hidden_layers=1,
)

In [4]:
import qewton.algorithms.building_blocks.array_operations as array_ops
import inspect

print([
    name for name, obj in inspect.getmembers(array_ops, inspect.isclass)
])

['Annotated', 'AxesDim', 'ConcatNode', 'ConcatVariables', 'DEFAULT_DL_BACKEND', 'DataConfiguration', 'DeepLearningBackend', 'EllipsisAxes', 'EllipsisType', 'FeatureAxes', 'Flatten', 'GetShapeNode', 'InputPort', 'NO_DEFAULT', 'Narrow', 'Node', 'OutputPort', 'Port', 'Repeat', 'Reshape', 'ReshapeAtDim', 'SetItem', 'Slice', 'SplitVariables', 'Squeeze', 'Unflatten', 'Unsqueeze', 'Variable']


In [5]:
import inspect

from qewton.algorithms.building_blocks.array_operations import (
    DataConfiguration,
    Node,
    Unsqueeze,
    Squeeze,
)

print("=== DataConfiguration ===")
print(inspect.getsource(DataConfiguration))

print("=== Unsqueeze ===")
print(inspect.getsource(Unsqueeze))

print("=== Squeeze ===")
print(inspect.getsource(Squeeze))

print("=== Node ===")
print(inspect.getsource(Node))

=== DataConfiguration ===
class DataConfiguration(Serializable):
    """A *DataConfiguration* describes the expected structure of the data,
    including the axes and their dimensions, as well as the data type.
    It is used to ensure that the data being passed through the graph
    matches the expected format for different methods/algorithms.

    Args:
        *axes (Axes | EllipsisAxes): The axes that describe the
            structure of the data.
        dtype (_type_, optional): The datatype used in this
            configuration. Defaults to None.
    """

    def __init__(self, *axes: Axes | EllipsisAxes, dtype=None):
        self.axes = axes
        self.dtype = dtype
        assert (
            len([f for f in axes if isinstance(f, FeatureAxes)]) <= 1
        ), "A DataConfig can have at most one FeatureAxes."

    @classmethod
    def empty(cls):
        """Builds an empty data configuration with only an ellipsis axis.
        This can be used as a placeholder when no spec

In [6]:
branch_input_port = data_loader.get_output_port(domain_var)

print("DataLoader output port:")
print(branch_input_port)

print("\nCNN branch port:")
print(model.branch_port)

print("\nDataLoader port attributes:")
print(vars(branch_input_port))

print("\nCNN branch port attributes:")
print(vars(model.branch_port))
import inspect

print(inspect.getsource(qewton.algorithms.CNNDeepONet))

sample = data["domain_mask"] if "data" in globals() else None


DataLoader output port:

CNN branch port:

DataLoader port attributes:
{'_static_data_configuration': BatchAxes([27]), GeometryAxes([100, 50, 25]), FeatureAxes([1]), 'node': <qewton.data.dataloaders.base.DataLoader object at 0x7f318132b2e0>, 'name': 'domain', '_value': None, 'value_persists': False}

CNN branch port attributes:
{'_static_data_configuration': BatchAxes([None]), FeatureAxes([1]), ..., 'node': <qewton.algorithms.dl_models.deeponet.deeponet_cnn.CNNDeepONet object at 0x7f31814eded0>, 'name': 'x', '_value': <class 'qewton.graphs.nodes.NO_DEFAULT'>, 'value_persists': False, 'default': <class 'qewton.graphs.nodes.NO_DEFAULT'>}
class CNNDeepONet(DeepONet[TensorType]):
    """A DeepONet architecture that uses fully connected networks (FCNs)
    for the trunk and a convolutional neural network (CNN) for the branch.
    The merge node can be configured to handle different output strategies,
    allowing for flexibility in how the outputs of the branch and trunk networks
    are co

In [8]:
constraint = qewton.constraints.MSEConstraint()

computation_graph = qewton.Graph()
transpose_node = qewton.bb.Transpose([0, 4, 1, 2, 3])
# Branch input
computation_graph.connect(
    data_loader.get_output_port(domain_var),
    transpose_node,
)

computation_graph.connect(
    transpose_node,
    model.branch_port,
)

# Trunk input
computation_graph.connect(
    data_loader.get_output_port(x_var),
    model.trunk_port,
)

# Model prediction
computation_graph.connect(
    model,
    constraint.input_1,
)

# Reference temperature
computation_graph.connect(
    data_loader.get_output_port(temperature_var),
    constraint.input_2,
)

In [9]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[computation_graph],
    training_objectives=[constraint],
    device="cuda:0",
)

trainer.run()

/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning:

Can't initialize NVML

Optimization Phase 1:   7%|▋         | 163/2500 [00:24<06:07,  6.36it/s, loss=3.95] 

KeyboardInterrupt: 

In [ ]:
temperature_config = (
    data_loader.get_output_port(temperature_var)
    .get_data_configuration(computation_graph)
)

layout = computation_graph.visualize(
    model.output_ports[0],
    reference=data_loader.get_output_port(temperature_var),
    prediction_config=temperature_config,
    mode=qewton.optim.EvaluationPhase.TEST,
)

qewton.visualization.Figure(layout).show()

TypeError: conv3d() received an invalid combination of arguments - got (_ProtocolMeta, groups=int, dilation=tuple, stride=tuple, padding=tuple, bias=Tensor, weight=Tensor), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, tuple of ints padding = 0, tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the keywords were incorrect: groups, dilation, stride, padding, bias
 * (Tensor input, Tensor weight, Tensor bias = None, tuple of ints stride = 1, str padding = "valid", tuple of ints dilation = 1, int groups = 1)
      didn't match because some of the keywords were incorrect: groups, dilation, stride, padding, bias
